## Step 8: Fine-Tuned Transfer Learning

In this step, transfer learning is enhanced by fine-tuning the pretrained MobileNetV2 model. Instead of keeping all pretrained layers frozen, the top layers are unfrozen and retrained on the pneumonia dataset.

This allows the model to adapt high-level features to domain-specific patterns present in chest X-ray images, improving performance and generalization.

Fine-tuning bridges the gap between generic image features and specialized medical imaging features, leading to better classification accuracy.


In [21]:
#  Import libraries

import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

In [22]:
#  Load dataset

df = pd.read_csv("/kaggle/input/datasets/sairasagnak/balanced-data-rsna/balanced_data.csv")
print(df.shape)

(12024, 8)


In [23]:
#  Fix paths

DATA_DIR = "/kaggle/input/datasets/iamtapendu/rsna-pneumonia-processed-dataset"
IMAGE_DIR = os.path.join(DATA_DIR, "Training", "Images")

df['image_path'] = df['patientId'].apply(lambda x: os.path.join(IMAGE_DIR, x + ".png"))
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)

In [24]:
#  Preprocessing

IMG_SIZE = 128

def preprocess_image(path):
    img = cv2.imread(path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img

In [25]:
#  Prepare dataset

df = df.sample(4000, random_state=42)

images = []
labels = []

for _, row in df.iterrows():
    img = preprocess_image(row['image_path'])
    images.append(img)
    labels.append(row['Target'])

X = np.array(images)
y = np.array(labels)

print(X.shape, y.shape)

(4000, 128, 128, 3) (4000,)


In [26]:
#  Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
# Load pretrained model (FREEZE COMPLETELY)

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

2026-05-03 09:40:19.898575: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [36]:
# Data augmentation

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [38]:
#  Fine-tune top layers

base_model.trainable = True

# Freeze lower layers, train top layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

In [40]:
#  Build model

model = models.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.6),
    layers.Dense(1, activation='sigmoid')
])

In [43]:
# Compile

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [44]:
#  Train

history = model.fit(
    X_train, y_train,
    epochs=8,
    validation_data=(X_test, y_test)
)

Epoch 1/8
100/100 ━━━━━━━━━━━━━━━━━━━━ 54s 433ms/step - accuracy: 0.6459 - loss: 0.8190 - val_accuracy: 0.8250 - val_loss: 0.3933
Epoch 2/8
100/100 ━━━━━━━━━━━━━━━━━━━━ 42s 422ms/step - accuracy: 0.7099 - loss: 0.6542 - val_accuracy: 0.7837 - val_loss: 0.4500
Epoch 3/8
100/100 ━━━━━━━━━━━━━━━━━━━━ 42s 416ms/step - accuracy: 0.7190 - loss: 0.6415 - val_accuracy: 0.8000 - val_loss: 0.4362
Epoch 4/8
100/100 ━━━━━━━━━━━━━━━━━━━━ 43s 428ms/step - accuracy: 0.7420 - loss: 0.5891 - val_accuracy: 0.8037 - val_loss: 0.4372
Epoch 5/8
100/100 ━━━━━━━━━━━━━━━━━━━━ 47s 471ms/step - accuracy: 0.7387 - loss: 0.5963 - val_accuracy: 0.8313 - val_loss: 0.3990
Epoch 6/8
100/100 ━━━━━━━━━━━━━━━━━━━━ 42s 422ms/step - accuracy: 0.7501 - loss: 0.5532 - val_accuracy: 0.8213 - val_loss: 0.3953
Epoch 7/8
100/100 ━━━━━━━━━━━━━━━━━━━━ 47s 473ms/step - accuracy: 0.7567 - loss: 0.5285 - val_accuracy: 0.8087 - val_loss: 0.4526
Epoch 8/8
100/100 ━━━━━━━━━━━━━━━━━━━━ 42s 420ms/step - accuracy: 0.7698 - loss: 0.5154 - 

In [46]:
#  Evaluate

loss, accuracy = model.evaluate(X_test, y_test)

print("Fine-Tuned Accuracy:", accuracy)

25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 216ms/step - accuracy: 0.8272 - loss: 0.4105
Fine-Tuned Accuracy: 0.8162500262260437


## Observations

* The fine-tuned transfer learning model achieved an accuracy of approximately 82%, outperforming all previous models.
* Freezing the pretrained layers and applying data augmentation significantly improved generalization and reduced overfitting.
* The validation accuracy remained stable across epochs, indicating a well-balanced learning process.
* Compared to earlier experiments, controlled training strategies proved more effective than increasing model complexity.
* The results demonstrate the effectiveness of transfer learning when combined with proper regularization and data augmentation.

This step highlights that successful deep learning models depend not only on architecture but also on appropriate training strategies and data handling.
